# Phi-4 Reasoning Inference with the OpenAI SDK (formerly GitHub Models)

> **Note:** GitHub Models was retired on July 30, 2026 ([GitHub changelog](https://github.blog/changelog/2026-07-30-github-models-is-now-retired/)), and the Azure AI Inference beta SDK (`azure-ai-inference`) was [retired on August 26, 2026](https://learn.microsoft.com/azure/foundry-classic/foundry-models/supported-languages). This notebook now uses the [OpenAI SDK](https://github.com/openai/openai-python) with an OpenAI-compatible endpoint:
>
> - **Microsoft Foundry** (cloud, requires an Azure subscription): set `OPENAI_BASE_URL` to `https://<resource>.openai.azure.com/openai/v1/`, `OPENAI_API_KEY` to your resource key and `MODEL` to your deployment name. See [Migrate from Azure AI Inference SDK to OpenAI SDK](https://learn.microsoft.com/azure/foundry/how-to/model-inference-to-openai-migration).
> - **Foundry Local** (on-device, no Azure subscription needed): set `OPENAI_BASE_URL` to `http://localhost:<port>/v1` (port from `foundry server status`), `OPENAI_API_KEY` to any non-empty value and `MODEL` to the model ID from `foundry model list --loaded --verbose`. See [Foundry Local](https://learn.microsoft.com/azure/foundry-local/).

This notebook demonstrates inference with Microsoft's Phi-4-reasoning model through an OpenAI-compatible endpoint.

## Setup and Configuration

First, we'll configure our environment and install necessary dependencies.

In [ ]:
# Install required packages
!pip install openai python-dotenv

### Setting up the local.env File

Before running this notebook, you need to create a `local.env` file in the same directory as this notebook with the following variables:

```
# Microsoft Foundry: https://<resource>.openai.azure.com/openai/v1/
# Foundry Local:     http://localhost:<port>/v1  (port from `foundry server status`)
OPENAI_BASE_URL=your_endpoint_here

# Microsoft Foundry: your resource key. Foundry Local: any non-empty value, e.g. none
OPENAI_API_KEY=your_api_key_here

# Microsoft Foundry: your deployment name, e.g. a deployment of Phi-4-reasoning
# Foundry Local: the model ID from `foundry model list --loaded --verbose`
MODEL=your_model_here
```

**Instructions:**

1. Create a new file named `local.env` in the same folder as this notebook
2. Add the three environment variables shown above
3. Replace the placeholder values with your endpoint, key and model
4. You can optionally use Phi-4-mini-reasoning for a smaller model

**Note:** Keep your API key secure and don't share it in public repositories or notebooks.

## Load Environment Variables

We'll load our environment variables from the `local.env` file which contains our endpoint, key and model information.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# Load variables from local.env file
load_dotenv('local.env')

# Access the environment variables - using values from local.env file
endpoint = os.getenv("OPENAI_BASE_URL")
token = os.getenv("OPENAI_API_KEY")
model = os.getenv("MODEL")

if not endpoint or not token or not model:
    raise ValueError("OPENAI_BASE_URL, OPENAI_API_KEY and MODEL must be set in the local.env file.")

client = OpenAI(base_url=endpoint, api_key=token)

print(f"Endpoint: {endpoint}")
print(f"Model: {model}")
print(f"Token available: {'Yes' if token else 'No'}")

## Helper Functions for Model Inference

Let's create helper functions to call the chat completions API through the OpenAI SDK.

In [ ]:
def generate_chat_completion(messages, model_id=model, temperature=0.7, max_tokens=1000):
    """Generate a completion using the chat completions API"""
    try:
        response = client.chat.completions.create(
            model=model_id,
            messages=messages,
            temperature=temperature,
            max_tokens=max_tokens,
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Full error details: {str(e)}")
        return f"Error during API call: {str(e)}"

def generate_completion(prompt, model_id=model, temperature=0.7, max_tokens=10000):
    """Generate a completion for a single user prompt"""
    return generate_chat_completion([{"role": "user", "content": prompt}], model_id, temperature, max_tokens)

def format_conversation(messages):
    """Format a conversation for the model"""
    # For chat completion API, we'll just return the messages directly
    return messages

## Example 1: How many strawberries for 9 r's?

Let's run our first inference example asking about strawberries and r's.

In [ ]:
example1_messages = [
    {"role": "system", "content": "You are a helpful AI assistant that answers questions accurately and concisely."},
    {"role": "user", "content": "How many strawberries do I need to collect 9 r's?"}
]

print("Messages:")
for msg in example1_messages:
    print(f"{msg['role']}: {msg['content']}")
print("\nGenerating response...\n")

# Use the new chat completion function directly with the messages
response1 = generate_chat_completion(example1_messages)
print("Response:")
print(response1)

### Analysis of Example 1

In this example, the model needs to understand that the word "strawberry" contains two 'r' letters. Therefore, to collect 9 'r's, you would need 5 strawberries (with 10 'r's total), or 4.5 strawberries to get exactly 9 'r's.

Let's see how the Phi-4-reasoning model handles this problem.

## Example 2: Solving a Riddle

Now let's try a more complex example - a pattern recognition riddle with multiple examples.

In [ ]:
example2_messages = [
    {"role": "system", "content": "You are a helpful AI assistant that solves riddles and finds patterns in sequences."},
    {"role": "user", "content": "I will give you a riddle to solve with a few examples, and something to complete at the end"},
    {"role": "user", "content": "nuno Δημήτρης evif Issis 4"},
    {"role": "user", "content": "ntres Inez neves Margot 4"},
    {"role": "user", "content": "ndrei Jordan evlewt Μαρία 9"},
    {"role": "user", "content": "nπέντε Kang-Yuk xis-ytnewt Nubia 21"},
    {"role": "user", "content": "nπέντε Κώστας eerht-ytnewt Μανώλης 18"}, 
    {"role": "user", "content": "nminus one-point-two Satya eno Bill X."},
    {"role": "user", "content": "What is a likely completion for X that is consistent with examples above?"}
]

print("Messages:")
for msg in example2_messages:
    print(f"{msg['role']}: {msg['content'][:50]}...")
print("\nGenerating response...\n")

response2 = generate_chat_completion(example2_messages, temperature=0.2, max_tokens=10000)
print("Response:")
print(response2)

### Analysis of Example 2

This riddle requires recognizing complex patterns across multiple languages and numerical representations. Let's break down what the model needs to understand:

1. Identify reverse spelling in words like "evif" (five)
2. Recognize numbers in different languages (e.g., "uno" in Spanish, "πέντε" in Greek)
3. Find the relationship between the numbers and the final digit

The pattern seems to involve mathematical operations between the values represented in different languages and formats.

## Experimenting with Different Parameters

Let's try the second example again with different temperature settings to see how it affects the model's response.

In [ ]:
# Try with a higher temperature for more creative responses
response_creative = generate_chat_completion(example2_messages, temperature=0.9, max_tokens=20000)
print("Response with higher temperature (0.9):")
print(response_creative)

## Create Your Own Example

You can create your own examples to test the model's reasoning capabilities. Try modifying the prompts or creating entirely new scenarios below.

In [ ]:
# Define your custom prompt here
custom_messages = [
    {"role": "system", "content": "You are a helpful AI assistant that can solve complex problems."},
    {"role": "user", "content": "Your custom prompt here"}
]

# Uncomment the lines below to run your custom prompt
# custom_response = generate_chat_completion(custom_messages)
# print("Response to custom prompt:")
# print(custom_response)

## Conclusion

This notebook demonstrated how to call the Phi-4-reasoning model through an OpenAI-compatible endpoint (Microsoft Foundry or Foundry Local) to solve logical problems and riddles.

Key takeaways:
1. Configuring the endpoint, key and model through environment variables
2. Formatting prompts for optimal inference
3. Managing model parameters like temperature to control response variation
4. Testing the model's reasoning capabilities on different types of problems

Remember to keep your API key secure and not share it in public repositories or notebooks.